# EVASPA Edge benchmark

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

In [ ]:
from evaspa.common import io
from evaspa.evaspa.edge import Edge

Plot functions

In [ ]:
def plot(
    var: npt.NDArray,
    lst: npt.NDArray,
    edges: list[Edge] | None = None,
    points: list[tuple[npt.NDArray, npt.NDArray]] | None = None,
    params: list[str] | None = None,
    var_name: str | None = None,
    out: str | None = None,
):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    _ = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    # plot all points
    ax.scatter(var, lst, s=10, alpha=1, color="moccasin", clip_on=False)

    # Fix limits
    xmin = np.nanmin(var) - 0.05
    xmax = np.nanmax(var) + 0.05
    ymin = np.nanmin(lst) - 5
    ymax = np.nanmax(lst) + 5
    ax.set_xlim([xmin, xmax])
    ax.set_ylim([ymin, ymax])

    # plot edge
    if edges is not None:
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)
        var_arr = np.linspace(var_min, var_max, num=1000)
        for edge in edges:
            edge_arr = edge.get(var_arr)

            ax.plot(
                var_arr,
                edge_arr,
                color="black",
                linewidth=3,
                zorder=1,
            )
    if points is not None:
        for pts in points:
            x_pts = pts[0]
            y_pts = pts[1]
            ax.scatter(x_pts, y_pts, color="red", marker="x")

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    # Save
    if edges is not None:
        plot_title = "EF"

        if params is not None and len(params) == 3:
            plot_title = f"LST-{var_name} \n EF {params[0]} interval={params[1]} points={params[2]}"
        ax.set_title(plot_title, fontsize=10)
        if out is not None and params is not None and len(params) == 3:
            filename = (
                f"ef_{params[0]}_interval_{params[1]}_points_{params[2]}.png"
            )
            plt.savefig(os.path.join(out, filename))

    plt.show()

### Read data

In [ ]:
src_path = os.path.dirname(os.path.dirname(os.getcwd()))
data = io.read_data_from_file(
    os.path.join(src_path, "tests", "data", "modis_test_geo.tif")
)
lst = data["lst"]
fcover = data["fcover"]
ndvi = data["ndvi"]
albedo = data["albedo"]

In [ ]:
# Output dir
output_data = os.path.join(os.getcwd(), "out")

In [ ]:
def run(var_name, var, lst, position, models, intervals, points):
    """
    Method to run several edges, with interval_nb and nb_points
    """
    out = os.path.join(output_data, var_name, position)
    os.makedirs(out, exist_ok=True)
    for name_model in models:
        use_breakpoint = False
        name = name_model
        if m := re.match(r"(.*)_bk", name):
            name = m.group(1)
            use_breakpoint = True
        for interval_nb in intervals:
            for nb_points in points:
                config = {
                    "position": position,
                    "interval_type": "density",
                    "interval_nb": interval_nb,
                    "nb_points": nb_points,
                    "selection": "median",
                    "use_breakpoint": use_breakpoint,
                }
                edge = Edge.create(name, config)
                edge.fit(var, lst)
                pts = edge.get_points(var, lst)
                plot(
                    var,
                    lst,
                    edges=[edge],
                    points=[pts],
                    params=[
                        f"{name}{'_bk' if use_breakpoint else ''}",
                        interval_nb,
                        nb_points,
                    ],
                    var_name=var_name,
                    out=out,
                )

### Test LST-Fcover

Plot

In [ ]:
plot(fcover, lst, var_name="fcover")

Top edge:  

- edges types: flat, linear regression &  parabolic
- number of intervals: 20, 50
- number of points: 20, 100, 250 & 1000 


In [ ]:
run(
    "fcover",
    fcover,
    lst,
    position="top",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    intervals=[20, 50],
    points=[20, 100, 250, 1000],
)

Bottom edge:  

- edges types: flat, linear regression &  parabolic
- number of intervals: 20, 50
- number of points: 20, 100, 250 & 1000 

In [ ]:
run(
    "fcover",
    fcover,
    lst,
    position="bottom",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    intervals=[20, 50],
    points=[20, 100, 250, 1000],
)

### Test LST-NDVI

Plot

In [ ]:
plot(ndvi, lst, var_name="ndvi")

Top edge:  

- edges types: flat, linear regression &  parabolic
- number of intervals: 20, 50
- number of points: 20, 100, 250 & 1000 


In [ ]:
run(
    "ndvi",
    ndvi,
    lst,
    position="top",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    intervals=[20, 50],
    points=[20, 100, 250, 1000],
)

Bottom edge:  

- edges types: flat, linear regression &  parabolic
- number of intervals: 20, 50
- number of points: 20, 100, 250 & 1000 

In [ ]:
run(
    "ndvi",
    ndvi,
    lst,
    position="bottom",
    models=["FlatRegressionEdge", "LinearEdge", "ParabolicEdge"],
    intervals=[20, 50],
    points=[20, 100, 250, 1000],
)

### Test LST-albedo

Plot

In [ ]:
plot(albedo, lst, var_name="albedo")

Top edge:  

- edges types: 7 edges types to test for both wet and dry edge: flat, linear regression, linear regression (considering only albedo after breakpoint), parabolic, linear regression on albedo upper than the break point, linear regression on albedo upper than the break point and flat on lower albedos, double linear regression on albedo upper and lower than the break point
- number of intervals: 20, 50
- number of points: 20, 100, 250 & 1000 


In [ ]:
run(
    "albedo",
    albedo,
    lst,
    position="top",
    models=[
        "FlatRegressionEdge",
        "LinearEdge",
        "LinearEdge_bk",
        "ParabolicEdge",
        "ThresholdLinearEdge",
        "FlatLinearEdge",
        "DoubleLinearEdge",
    ],
    intervals=[20, 50],
    points=[20, 100, 250, 1000],
)

Bottom edge:  

- edges types: 7 edges types to test for both wet and dry edge: flat, linear regression, linear regression (considering only albedo after breakpoint), parabolic, linear regression on albedo upper than the break point, linear regression on albedo upper than the break point and flat on lower albedos, double linear regression on albedo upper and lower than the break point
- number of intervals: 20, 50
- number of points: 20, 100, 250 & 1000 

In [ ]:
run(
    "albedo",
    albedo,
    lst,
    position="bottom",
    models=[
        "FlatRegressionEdge",
        "LinearEdge",
        "LinearEdge_bk",
        "ParabolicEdge",
        "ThresholdLinearEdge",
        "FlatLinearEdge",
        "DoubleLinearEdge",
    ],
    intervals=[20, 50],
    points=[20, 100, 250, 1000],
)